# Duckietown Explore Retrain (Stable Path)

Uses the existing stable trainer: `/content/leworldduckie/src/train.py`.


In [ ]:
# Config — exploratory retrain, planned fs6 config (see docs/memory + T6 gate)
REPO_URL = 'https://github.com/giuliovv/leworldduckie.git'
DATA_URL = 'https://leworldduckie-public-data.s3.us-east-1.amazonaws.com/duckie_explore.h5'
DATA_LOCAL = '/content/duckie_explore.h5'
RUN_ID = 'duckie_explore_fs6'
# Artifact destination — the leworldduckie-colab key is scoped to this bucket
# (ERP AWS account). Synced back to s3://leworldduckie/training/runs/ afterwards.
ARTIFACT_BUCKET = 'leworldduckie-colab-artifacts'
EPOCHS = 50
BATCH_SIZE = 128

# Model/training config — MUST stay in sync with the eval side:
# t6_eval.py and mpc_controller.py read the same ACTION_SCALE env var.
TRAIN_ENV = {
    'FRAMESKIP':    '6',
    'N_PREDS':      '4',
    'SIGREG_W':     '0.1',
    'ACTION_SCALE': '5.3',
}


In [ ]:
# AWS credentials — REQUIRED for checkpoint upload to S3.
# Preferred: add COLAB secrets AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY
# (key icon in the left sidebar), then this cell picks them up.
# Fallback: interactive prompt.
import os
try:
    from google.colab import userdata
    os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
    os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
    print('AWS credentials loaded from Colab secrets')
except Exception:
    import getpass
    os.environ['AWS_ACCESS_KEY_ID'] = getpass.getpass('AWS_ACCESS_KEY_ID: ')
    os.environ['AWS_SECRET_ACCESS_KEY'] = getpass.getpass('AWS_SECRET_ACCESS_KEY: ')
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'

import subprocess
subprocess.check_call(['bash','-lc','pip -q install boto3'])
import boto3
ident = boto3.client('sts').get_caller_identity()
print('S3 upload identity OK:', ident['Arn'])


In [ ]:
# Setup
import os, subprocess
subprocess.check_call(['bash','-lc','python3 -m pip install -U pip'])
subprocess.check_call(['bash','-lc','pip install torch torchvision einops boto3 h5py matplotlib'])
subprocess.check_call(['bash','-lc', f'git clone --depth 1 {REPO_URL} /content/leworldduckie || true'])
if not os.path.exists(DATA_LOCAL):
    subprocess.check_call(['bash','-lc', f'wget -O {DATA_LOCAL} "{DATA_URL}"'])
print('data:', DATA_LOCAL, os.path.getsize(DATA_LOCAL))


In [ ]:
# Train (stable src/train.py path)
import os, subprocess
env = os.environ.copy()
env['DATA_PATH'] = DATA_LOCAL
env['S3_DATA_KEY'] = 'data/duckie_explore.h5'
env['S3_BUCKET'] = ARTIFACT_BUCKET
env['N_EPOCHS'] = str(EPOCHS)
env['BATCH_SIZE'] = str(BATCH_SIZE)
env.update(TRAIN_ENV)
cmd = f'cd /content/leworldduckie && python3 src/train.py --run-id {RUN_ID} --epochs {EPOCHS} 2>&1 | tee /content/train_duckie.log'
print(cmd)
print('env:', {k: env[k] for k in ('FRAMESKIP','N_PREDS','SIGREG_W','ACTION_SCALE')})
ret = subprocess.run(['bash','-lc',cmd], env=env)
if ret.returncode != 0:
    raise RuntimeError(f'training failed: {ret.returncode}; see /content/train_duckie.log')


In [ ]:
# Verify checkpoint landed on S3 — do NOT trust "training finished" alone.
import boto3
s3 = boto3.client('s3')
prefix = f'training/runs/{RUN_ID}/'
resp = s3.list_objects_v2(Bucket=ARTIFACT_BUCKET, Prefix=prefix)
keys = [o['Key'] for o in resp.get('Contents', [])]
print('\n'.join(keys) or '(nothing)')
assert any(k.endswith('checkpoint_best.pt') for k in keys), (
    f'checkpoint_best.pt NOT on S3 under {prefix} — '
    'copy /tmp/run_*/checkpoint_best.pt out of this runtime NOW before it is recycled')
print(f'\nOK: s3://{ARTIFACT_BUCKET}/{prefix}checkpoint_best.pt')
